In [1]:
import pandas as pd 
import numpy as np

# Loading the data

In [2]:
df = pd.read_csv('../data/cleaning_data_for_recommender_system/cleaned_movies.csv')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_33908\1226104732.py:1: DtypeWarning: Columns (0: cast, 1: director) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/cleaning_data_for_recommender_system/cleaned_movies.csv')


In [3]:
df.head()

,id,title,tagline,overview,release_date,runtime,genres,original_language,cast,director,production_companies,revenue,budget,vote_count,vote_average,popularity,status
0,615656,Meg 2: The Trench,Back for seconds.,An exploratory dive into the deepest depths of...,2023-08-02,116.0,Action-Science Fiction-Horror,en,Jason Statham as Jonas Taylor|Wu Jing as |Shuy...,Ben Wheatley,Apelles Entertainment-Warner Bros. Pictures-di...,3.520565e+08,129000000.0,1365.0,7.079,8763.998,Released
1,758323,The Pope's Exorcist,Inspired by the actual files of Father Gabriel...,Father Gabriele Amorth Chief Exorcist of the V...,2023-04-05,103.0,Horror-Mystery-Thriller,en,Russell Crowe as Father Gabriele Amorth|Daniel...,Julius Avery,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,6.567582e+07,18000000.0,545.0,7.433,5953.227,Released
2,533535,Deadpool & Wolverine,Come together.,A listless Wade Wilson toils away in civilian ...,2024-07-24,128.0,Action-Comedy-Science Fiction,en,Ryan Reynolds as Wade Wilson / Deadpool|Hugh J...,Shawn Levy,Marvel Studios-Maximum Effort-21 Laps Entertai...,1.326387e+09,200000000.0,3749.0,7.765,5410.496,Released
3,667538,Transformers: Rise of the Beasts,Unite or fall.,When a new threat capable of destroying the en...,2023-06-06,127.0,Action-Adventure-Science Fiction,en,Anthony Ramos as Noah Diaz|Dominique Fishback ...,Steven Caple Jr.,Skydance-Paramount-di Bonaventura Pictures-Bay...,4.070455e+08,200000000.0,1007.0,7.340,5409.104,Released
4,693134,Dune: Part Two,Long live the fighters.,Follow the mythic journey of Paul Atreides as ...,2024-02-27,167.0,Science Fiction-Adventure,en,Timothée Chalamet as Paul Atreides|Zendaya as ...,Denis Villeneuve,Legendary Pictures,6.838137e+08,190000000.0,2770.0,8.300,4742.163,Released


# Basic data cleaning

In [4]:
min_value = 5

df
df = df.fillna('').replace({'Unknown': '', 'No Overview': '', 'No Tagline': '', 'No Collection': ''})
df['release_date'] = pd.to_datetime(df['release_date'], format='%Y-%m-%d', errors='coerce')
df['year'] = df['release_date'].dt.year
df.loc[df['year'] < 1854, 'year'] = np.nan
df= df[df['vote_count']>=min_value].reset_index(drop=True)
df.head()

,id,title,tagline,overview,release_date,runtime,genres,original_language,cast,director,production_companies,revenue,budget,vote_count,vote_average,popularity,status,year
0,615656,Meg 2: The Trench,Back for seconds.,An exploratory dive into the deepest depths of...,2023-08-02,116.0,Action-Science Fiction-Horror,en,Jason Statham as Jonas Taylor|Wu Jing as |Shuy...,Ben Wheatley,Apelles Entertainment-Warner Bros. Pictures-di...,3.520565e+08,129000000.0,1365.0,7.079,8763.998,Released,2023.0
1,758323,The Pope's Exorcist,Inspired by the actual files of Father Gabriel...,Father Gabriele Amorth Chief Exorcist of the V...,2023-04-05,103.0,Horror-Mystery-Thriller,en,Russell Crowe as Father Gabriele Amorth|Daniel...,Julius Avery,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,6.567582e+07,18000000.0,545.0,7.433,5953.227,Released,2023.0
2,533535,Deadpool & Wolverine,Come together.,A listless Wade Wilson toils away in civilian ...,2024-07-24,128.0,Action-Comedy-Science Fiction,en,Ryan Reynolds as Wade Wilson / Deadpool|Hugh J...,Shawn Levy,Marvel Studios-Maximum Effort-21 Laps Entertai...,1.326387e+09,200000000.0,3749.0,7.765,5410.496,Released,2024.0
3,667538,Transformers: Rise of the Beasts,Unite or fall.,When a new threat capable of destroying the en...,2023-06-06,127.0,Action-Adventure-Science Fiction,en,Anthony Ramos as Noah Diaz|Dominique Fishback ...,Steven Caple Jr.,Skydance-Paramount-di Bonaventura Pictures-Bay...,4.070455e+08,200000000.0,1007.0,7.340,5409.104,Released,2023.0
4,693134,Dune: Part Two,Long live the fighters.,Follow the mythic journey of Paul Atreides as ...,2024-02-27,167.0,Science Fiction-Adventure,en,Timothée Chalamet as Paul Atreides|Zendaya as ...,Denis Villeneuve,Legendary Pictures,6.838137e+08,190000000.0,2770.0,8.300,4742.163,Released,2024.0


In [5]:
df.info

<bound method DataFrame.info of             id                             title  \
0       615656                 Meg 2: The Trench   
1       758323               The Pope's Exorcist   
2       533535              Deadpool & Wolverine   
3       667538  Transformers: Rise of the Beasts   
4       693134                    Dune: Part Two   
...        ...                               ...   
103200  417514                   Herr von Bohlen   
103201  407739      Spring Break Zombie Massacre   
103202  337680                Travelling Warrior   
103203  260793                          Caudillo   
103204  414194                    Wild Playgirls   

                                                  tagline  \
0                                       Back for seconds.   
1       Inspired by the actual files of Father Gabriel...   
2                                          Come together.   
3                                          Unite or fall.   
4                                 Long

# Creating Soup

In [6]:
import re

In [7]:
crj = lambda x: ''.join(ch for ch in x.lower() if ch.isalnum())

df['genre_tokens'] = df['genres'].apply(lambda x: ' '.join([crj(g) for g in re.split(r'[|-]', x) if g]))
df['cast_tokens'] = df['cast'].apply(lambda x: ' '.join([crj(c.split(' as ', 1)[0]) for c in x.split('|') if c][:3]))
df['director_tokens'] = df['director'].apply(lambda x: ' '.join([crj(d) for d in x.split(',') if d]))


df['meta_soup'] = df['genre_tokens'] + ' ' + df['cast_tokens'] + ' ' + df['director_tokens']

df['plot_soup'] = df['overview'] + ' ' + df['tagline']



In [8]:
df.head()

,id,title,tagline,overview,release_date,runtime,genres,original_language,cast,director,...,vote_count,vote_average,popularity,status,year,genre_tokens,cast_tokens,director_tokens,meta_soup,plot_soup
0,615656,Meg 2: The Trench,Back for seconds.,An exploratory dive into the deepest depths of...,2023-08-02,116.0,Action-Science Fiction-Horror,en,Jason Statham as Jonas Taylor|Wu Jing as |Shuy...,Ben Wheatley,...,1365.0,7.079,8763.998,Released,2023.0,action sciencefiction horror,jasonstatham wujing shuyasophiacai,benwheatley,action sciencefiction horror jasonstatham wuji...,An exploratory dive into the deepest depths of...
1,758323,The Pope's Exorcist,Inspired by the actual files of Father Gabriel...,Father Gabriele Amorth Chief Exorcist of the V...,2023-04-05,103.0,Horror-Mystery-Thriller,en,Russell Crowe as Father Gabriele Amorth|Daniel...,Julius Avery,...,545.0,7.433,5953.227,Released,2023.0,horror mystery thriller,russellcrowe danielzovatto alexessoe,juliusavery,horror mystery thriller russellcrowe danielzov...,Father Gabriele Amorth Chief Exorcist of the V...
2,533535,Deadpool & Wolverine,Come together.,A listless Wade Wilson toils away in civilian ...,2024-07-24,128.0,Action-Comedy-Science Fiction,en,Ryan Reynolds as Wade Wilson / Deadpool|Hugh J...,Shawn Levy,...,3749.0,7.765,5410.496,Released,2024.0,action comedy sciencefiction,ryanreynolds hughjackman morenabaccarin,shawnlevy,action comedy sciencefiction ryanreynolds hugh...,A listless Wade Wilson toils away in civilian ...
3,667538,Transformers: Rise of the Beasts,Unite or fall.,When a new threat capable of destroying the en...,2023-06-06,127.0,Action-Adventure-Science Fiction,en,Anthony Ramos as Noah Diaz|Dominique Fishback ...,Steven Caple Jr.,...,1007.0,7.340,5409.104,Released,2023.0,action adventure sciencefiction,anthonyramos dominiquefishback petercullen,stevencaplejr,action adventure sciencefiction anthonyramos d...,When a new threat capable of destroying the en...
4,693134,Dune: Part Two,Long live the fighters.,Follow the mythic journey of Paul Atreides as ...,2024-02-27,167.0,Science Fiction-Adventure,en,Timothée Chalamet as Paul Atreides|Zendaya as ...,Denis Villeneuve,...,2770.0,8.300,4742.163,Released,2024.0,sciencefiction adventure,timothéechalamet zendaya rebeccaferguson,denisvilleneuve,sciencefiction adventure timothéechalamet zend...,Follow the mythic journey of Paul Atreides as ...


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from scipy.sparse import hstack


In [10]:
meta_value , plot_value = 0.6, 0.4

meta_matrix = normalize(CountVectorizer(token_pattern=r'\S+').fit_transform(df['meta_soup'])) 

plot_matrix = TfidfVectorizer( min_df=2,stop_words='english',max_features=30000,sublinear_tf=True).fit_transform(df['plot_soup']) 

matrix = normalize(hstack([meta_value * meta_matrix, plot_value * plot_matrix]).tocsr())

print('Matrix shape:', matrix.shape)

Matrix shape: (103205, 95796)


# Recommender system

In [11]:
import difflib
from sklearn.metrics.pairwise import linear_kernel

In [12]:
POP_weights = 0.25

titles_lower = df['title'].str.lower()
unique_titles = titles_lower.unique().tolist()
log_values = np.log10(df['vote_count'].values)

def recommend (title, n=10):
    matches = df.index[titles_lower == title.lower()]
    if len(matches) == 0:
        close = difflib.get_close_matches(title.lower(), unique_titles, n=5, cutoff=0.6)
        print(f'No exact match found for "{title}". Did you mean one of these? : {close}')
        return None
    idx = df.loc[matches , 'vote_count'].idxmax()
    sims = linear_kernel(matrix[idx], matrix).ravel()
    sims[matches] = -1

    candidates = np.argsort(sims)[::-1][:100]
    scores = sims[candidates] * (1 + POP_weights * log_values[candidates])
    top = candidates[np.argsort(scores)[::-1][:n]]

    out = df.loc[top, ['title', 'year', 'vote_average', 'vote_count']].copy()
    out['similarity'] = sims[top].copy()
    return out

# Trying it out

In [13]:
recommend('The Dark Knight', n=10)

,title,year,vote_average,vote_count,similarity
620,The Dark Knight Rises,2012.0,7.779,22124.0,0.696167
701,Batman Begins,2005.0,7.709,20851.0,0.599180
78369,Lamberto Quintero,1987.0,6.700,18.0,0.509525
90625,Segurança Nacional,2010.0,4.000,14.0,0.509525
57476,Black Rose,2014.0,2.500,17.0,0.500699
77913,Pups,1999.0,6.100,19.0,0.494697
55315,The Road Killers,1994.0,5.400,18.0,0.496663
22780,MYSTERIOUS FIGHTER Project A,2018.0,7.300,13.0,0.509525
59948,Massacre Mafia Style,1974.0,5.900,19.0,0.492698
52691,Crime in the Streets,1956.0,5.900,17.0,0.496888


In [14]:
recommend('interselar', n=10)

No exact match found for "interselar". Did you mean one of these? : ['interstellar', 'interstellar wars', 'winterset', 'intersect', 'winter war']


# Saving it 

In [15]:
import joblib

In [16]:
joblib.dump({'df': df[['title', 'year', 'genres', 'director', 'vote_average', 'vote_count']],
             'features': matrix},
            '../traditional_model/recommender_artifacts.joblib', compress=3)

['../traditional_model/recommender_artifacts.joblib']